# In-class exercise 6b — SOLUTION: Converting Tweets to matrix and Do fun things with them

## Remember the moral dictionary? From count method to frequency and inverse frequency

In the last session, we have pre-processed the American presidential candidates' tweets and applied a simple counting/frequency method to check out who appeals to which moral value. 

The moral dictionary (or any custom-made dictionary) is a fixed set of words. If you have a good theoritical foundation on why particularly these words need to be considered, it is a great idea - simple, easy to implement, deterministic and convincing. Any fancy algorithm won't necessary perform better in this case. 

But what if you don't have a good set of words to use, or that your research question is not conditioned on one specific angle? In this case, we can also convert the text data to "general" matrix. Imagine that instead of the 10 types of words in the MFD, you have one column that count **each individual English word**. Then, you can do fun things with it (such as machine learning, see the next class!) 

Of course, such a matrix would be extremely large and many of their words are completely uninformative (such as the word "the"). That's why we use all kinds of techniques trying to extract the "useful" information of a text, most importantly limiting to the top [insert dimension] N-grams and applying TF-IDF. 

In [1]:
# Import everything we need today 

# The basics
import scipy 
from pathlib import Path
import pandas as pd

# NLP tool kit
import nltk
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import wordnet
from string import punctuation
from nltk.corpus import stopwords
from nltk import word_tokenize, pos_tag

# Regular expressions
import re

# Language detection 
#import langdetect

# Loading
import fastparquet

# convert the tweets to embeddings using word2vec
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import numpy as np

# K-mean clustering 
from sklearn.cluster import KMeans

# Tfidf Vectoriser
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Our example the last time 
this_dir = Path(".")
df_path = this_dir / "data" / "cooked_data.parquet"
df = pd.read_parquet(df_path)

# let's quickly convert the column "processed text" in a single string instead of a list of words
import json

def join_with_space(lst):
    if isinstance(lst, str):   # list saved as a JSON string by fastparquet and read by pyarrow
        lst = json.loads(lst)
    return(" ".join(lst))

df['processed_text'] = df['processed_text'].apply(join_with_space)

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:997)>
[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [SSL: CERTIFICATE_VERIFY_FAILED] certificate
[nltk_data]     verify failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:997)>


### Exercise 1: Getting to know TF-IDF vectorizers

In this exercise, we will use the `TfidfVectorizer` from `sklearn` to convert the tweets into a matrix.

TFIDF is in its essence a counting method but with reweighting: it weights down the words that appear in many documents (e.g., "the", "is", "and") and weights up the words that appear in fewer documents (e.g., "election", "vote"). Let's first look at a small example: please take a look at the first 10 entries in our database. 

Run all the code blocks in this section. 


In [2]:
# Taking a look at the first 10 entries (they are from Trump 2016)
print(df['processed_text'][0:10])

index
0                                  land iowa speak soon
1                                                  time
3                                               # enjoy
5                                       mexico pay wall
8                                                 watch
9                                             # p enjoy
10       trump promise special session repeal obamacare
11                      yet evidence media rig election
12                                great job far contest
14    hillary advisers want avoid support israel tal...
Name: processed_text, dtype: object


In [3]:
# Using CountVectorizer
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(df['processed_text'][0:10])
print("Count Vectorizer Result:\n", count_matrix.toarray())

# Using TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'][0:10])
print("TF-IDF Vectorizer Result:\n", tfidf_matrix.toarray())

Count Vectorizer Result:
 [[0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 1 0 0 1 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 1 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0]]
TF-IDF Vectorizer Result:
 [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.5        0.
  0.         0.5        0.         0.         0.         0.
  0.         0.         0.         0.         0.5        0.5
  0.         0.

In [4]:
tfidf_matrix.shape

(10, 33)

In [5]:
# Looking at the words it take
print(count_vectorizer.get_feature_names_out())
print(tfidf_vectorizer.get_feature_names_out())

['advisers' 'avoid' 'contest' 'democrats' 'election' 'enjoy' 'evidence'
 'far' 'great' 'hillary' 'iowa' 'israel' 'job' 'land' 'media' 'mexico'
 'obamacare' 'pay' 'promise' 'repeal' 'rig' 'session' 'soon' 'speak'
 'special' 'support' 'talk' 'time' 'trump' 'wall' 'want' 'watch' 'yet']
['advisers' 'avoid' 'contest' 'democrats' 'election' 'enjoy' 'evidence'
 'far' 'great' 'hillary' 'iowa' 'israel' 'job' 'land' 'media' 'mexico'
 'obamacare' 'pay' 'promise' 'repeal' 'rig' 'session' 'soon' 'speak'
 'special' 'support' 'talk' 'time' 'trump' 'wall' 'want' 'watch' 'yet']


#### Questions to answer: 

1. Each of the matrices is 10x33. Why is there 10 vectors and 33 entries per vector in the matrix? 
2. Explain with your own world what an entry means: for example, what does a 1 in the 11st place in the first vector represent? 
3. Look at the first vector in both vectorizer. Why is the entries for "Iowa" and "land" 1 in the count vectorizer and 0.5 in the second vectorizer?  
4. Why do we want to use the TF-IDF vectorizer instead of the count vectorizer? 
Hint: imagine two scenarios: 
- Trump always finishes his tweet with "enjoy!"
- Trump tweets in a long and repeated manner, such as this tweet below: 


> “When will all of the ‘reporters’ who have received Noble Prizes for their work on Russia, Russia, Russia, only to have been proven totally wrong (and, in fact, it was the other side who committed the crimes), be turning back their cherished ‘Nobles’ so that they can be given…to the REAL REPORTERS & JOURNALISTS who got it right. I can give the Committee a very comprehensive list. When will the Noble Committee DEMAND the Prizes back, especially since they were gotten under fraud? The reporters and Lamestream Media knew the truth all along …Lawsuits should be brought against all, including the Fake News Organizations, to rectify this terrible injustice. For all of the great lawyers out there, do we have any takers? When will the Noble Committee Act? Better be fast!”



**Answers**

1. **10 vectors**: there is one row per document, and we vectorized the first 10 tweets. **33 entries**: there is one column per distinct word (token) in the vocabulary of these 10 tweets. The vectorizer found 33 different words (`get_feature_names_out()` lists them in alphabetical order). So the matrix has shape *(number of documents × size of the vocabulary)*.

2. Each entry *(i, j)* describes word *j* in tweet *i*. In the count vectorizer, it is the number of times word *j* appears in tweet *i*. The 11th position (index 10) is the feature `'iowa'`, so the 1 in the 11th place of the first vector means that **"iowa" appears once in the first tweet** ("land iowa speak soon"). The 0s mean that the other 29 words of the vocabulary do not appear in that tweet.

3. The first tweet has 4 words (*land, iowa, speak, soon*), each appearing once and only in this tweet among the 10. So they all have the same term frequency (1) and the same idf, hence the same raw tf-idf weight *w*. By default `TfidfVectorizer` normalizes each row to unit Euclidean length (`norm='l2'`): $\sqrt{4w^2}=1 \Rightarrow w = 1/\sqrt{4} = 0.5$. In the same way, the tweet "mexico pay wall" gets $1/\sqrt{3}\approx 0.577$ and the one-word tweets get 1.

4. Count vectors favour words that are frequent, even when they tell us nothing; tf-idf corrects this in two ways:
   - **idf** (inverse document frequency) gives a lower weight to words that appear in many documents. If Trump finishes every tweet with "enjoy!", "enjoy" appears everywhere, says nothing about the content of a particular tweet, and with tf-idf its weight goes down. Rare, specific words ("obamacare", "israel") get a higher weight.
   - **Length normalization**: each tweet vector has norm 1, so a long and repetitive tweet (Russia, Russia, Russia… Noble, Noble, Noble…) does not get huge values just because it is long. Tweets of different lengths can then be compared, which matters for distances and similarities (k-means, cosine). (Repeating a word still raises its *relative* weight within the tweet; `sublinear_tf=True` would dampen that further.)

### Exercise 2: implement and fine-tuning the vectorizer 

After the small illustration, we want to apply the TFIDF vectorizer immediately to the whole data base. The next cell contain some basic grammar for creating the TFIDF vectorizer and checking its shape.

We cut down to own biden's tweets to make it faster. Please run the 2 code blocks below.

In [6]:
# Cut to biden's tweets
df = df[df['candidate'] == 'biden']

In [7]:
# create another vectorizer 
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

#print some samples
print(tfidf_vectorizer.get_feature_names_out()[-20:])
print(tfidf_vectorizer.get_feature_names_out()[0:20])


# How long is this? 
tfidf_vectorizer.get_feature_names_out().shape

['yo' 'yom' 'york' 'yorknorth' 'you' 'youd' 'youll' 'young' 'younger'
 'youre' 'youth' 'youve' 'zero' 'zeta' 'zip' 'zl' 'zones' '보내세요' '한가위'
 '행복한']
['10' '100th' '10216donald' '1030am' '10th' '1215pm' '16th' '19th' '1st'
 '21st' '350pm' '3pm' '3rd' '47th' '48th' '4th' '50th' '5283nurses' '57th'
 '60']


(3897,)

Biden's 2k tweets gave us 3897 features. That looks quite long! Looking at the list of the features, there are some that we probably doesn't need: numbers that failed to be cleaned ('10' '100th' '10216donald' '1030am' '10th' '1215pm' '16th') or things that are typos ("yom"), foreign language ("행복한"). We want only meaningful words! 

To do this, we are going to fine-tune the TFIDF vectorizer so that it contains only the meaningful words. Please create a few different TF-idf vectorizes with the following options, using the document as the reference: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html. What do each critieria do?

- try the max_df and min_df options : set max_df to 0.9 and min_df to 0.01. What do you get?
- try the max_features options: set max_features to 500. What do you get?
- try n-grams (features with more than a word) while limiting the N of features in a certain way:set n grams to 1 or 2 what do you get? 
- try the vocabulary option and load in our moral dictionary from the last section. 

In [8]:
def describe(vectorizer, name):
    matrix = vectorizer.fit_transform(df['processed_text'])
    features = vectorizer.get_feature_names_out()
    print(f"--- {name}: matrix {matrix.shape}")
    print("first 15:", list(features[:15]))
    print("last 15 :", list(features[-15:]))
    return vectorizer

# max_df / min_df: drop words that appear in more than 90% / in less than 1% of the tweets
describe(TfidfVectorizer(max_df=0.9, min_df=0.01), "max_df=0.9, min_df=0.01")

# max_features: keep only the 500 most frequent words of the corpus
describe(TfidfVectorizer(max_features=500), "max_features=500")

# n-grams: words and pairs of consecutive words, limited to the 1000 most frequent
describe(TfidfVectorizer(ngram_range=(1, 2)), "ngram_range=(1, 2), no limit")
describe(TfidfVectorizer(ngram_range=(1, 2), max_features=1000), "ngram_range=(1, 2), max_features=1000")

# vocabulary: only count the words of the moral foundation dictionary (saved in exercise 6a)
mfd = pd.read_parquet(this_dir / "data" / "cooked_mfd.parquet")
moral_vocabulary = sorted(mfd['word'].unique())
moral_vectorizer = describe(TfidfVectorizer(vocabulary=moral_vocabulary, ngram_range=(1, 3)), "vocabulary=MFD")
moral_matrix = moral_vectorizer.transform(df['processed_text'])
print("share of Biden tweets with at least one moral word:", round((moral_matrix.getnnz(axis=1) > 0).mean(), 3))

--- max_df=0.9, min_df=0.01: matrix (2222, 312)
first 15: ['able', 'access', 'across', 'act', 'action', 'administration', 'ago', 'ahead', 'always', 'america', 'american', 'americans', 'another', 'anyone', 'around']
last 15 : ['win', 'wish', 'without', 'women', 'wont', 'word', 'work', 'workers', 'world', 'worst', 'would', 'year', 'years', 'yet', 'youre']
--- max_features=500: matrix (2222, 500)
first 15: ['able', 'access', 'across', 'act', 'action', 'actually', 'address', 'administration', 'afford', 'affordable', 'ago', 'ahead', 'allow', 'alone', 'already']
last 15 : ['without', 'women', 'wont', 'word', 'work', 'workers', 'world', 'worry', 'worst', 'would', 'write', 'year', 'years', 'yet', 'youre']
--- ngram_range=(1, 2), no limit: matrix (2222, 24325)
first 15: ['10', '100th', '100th anniversary', '10216donald', '10216donald trump', '1030am', '1030am edt', '10th', '10th anniversary', '1215pm', '1215pm edt', '16th', '16th street', '19th', '19th amendment']
last 15 : ['zero blame', 'zero

--- ngram_range=(1, 2), max_features=1000: matrix (2222, 1000)
first 15: ['able', 'able please', 'abuse', 'accept', 'access', 'accomplish', 'achieve', 'across', 'across country', 'across nation', 'across state', 'act', 'action', 'actually', 'address']
last 15 : ['worst possible', 'worth', 'would', 'write', 'wrong', 'year', 'years', 'years ago', 'years donald', 'yesterday', 'yet', 'youll', 'young', 'youre', 'youre able']


--- vocabulary=MFD: matrix (2222, 1440)
first 15: ['abhor', 'abstinance', 'abstinence', 'abuse', 'abuser', 'abusers', 'ache', 'achingly', 'acquiesce', 'acquiescent', 'addict', 'addiction', 'addictions', 'adhere', 'adherence']
last 15 : ['wholesome', 'wholesomely', 'wholesomeness', 'whore', 'whorehouse', 'whorehouses', 'wife', 'will', 'will rob', 'will share', 'wind', 'worship', 'wound', 'yogi', 'yogis']


share of Biden tweets with at least one moral word: 0.786


**What does each option do?** (results on Biden's 2,222 tweets, starting from 3,897 features)

- **`max_df=0.9, min_df=0.01`**: `max_df` drops the words that appear in more than 90% of the documents (too common to be informative, like corpus-specific stopwords). `min_df` drops the words that appear in less than 1% of the documents, i.e. fewer than about 22 tweets (typos, numbers like "10216donald", Korean words, very rare names). We go from 3,897 to **312** features, and they are all real, meaningful words. Floats are proportions of documents; integers would be absolute counts of documents.
- **`max_features=500`**: keeps only the 500 words with the highest frequency across the corpus. It sets the size of the matrix directly (500 columns), but it does not remove words that appear everywhere.
- **`ngram_range=(1, 2)`**: features are single words **and** pairs of consecutive words (bigrams such as "health care", "make sure", "soul nation"). This keeps some context and word order, but the number of features explodes (**24,325**, most of them seen only once). We have to combine it with `max_features` and/or `min_df` (e.g. 1,000 features with `max_features=1000`).
- **`vocabulary=...`**: we give the list of features ourselves, here the (lemmatized) moral foundation dictionary. The matrix then has exactly one column per MFD entry (1,440, plus bigrams/trigrams for expressions like "civil right"), and all other words are ignored. This is the "dictionary method" from last class, with tf-idf weights instead of raw counts (79% of Biden's tweets contain at least one MFD word).

#### To proceed

For our next step, create a vectorizer with max 2000 features, max_df = 0.9 and min_df = 0.005, and n-grams of 1 and 2. Check the shape of the resulting matrix and print out the last 20 features and the first 20 features. Also remove these specific words: 

`special_stop_words = ['trump', 'hillary', 'clinton', 'president', 'vote', 'election', 'amp', 'realdonaldtrump', 'donald', 'makeamericagreatagain', 'imwithher', 'gettyimages', 'us', 'one', 'new', 'now', 'like', 'day', 'time', 'see', 'want', 'go', 'know', "hillary clinton", 'joe', 'biden', 'democrat', 'democrats', 'democratic', 'america', 'american', 'people', 'country', 'think', 'great', 'donald', 'donald trump', 'vote', 'america'  ]`

In [9]:
special_stop_words = ['trump', 'hillary', 'clinton', 'president', 'vote', 'election', 'amp', 'realdonaldtrump', 'donald', 'makeamericagreatagain', 'imwithher', 'gettyimages', 'us', 'one', 'new', 'now', 'like', 'day', 'time', 'see', 'want', 'go', 'know', "hillary clinton", 'joe', 'biden', 'democrat', 'democrats', 'democratic', 'america', 'american', 'people', 'country', 'think', 'great', 'donald', 'donald trump', 'vote', 'america'  ]

tfidf_vectorizer = TfidfVectorizer(max_features=2000,
                                   max_df=0.9,
                                   min_df=0.005,
                                   ngram_range=(1, 2),
                                   stop_words=special_stop_words)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])

features = tfidf_vectorizer.get_feature_names_out()
print(tfidf_matrix.shape)
print(features[-20:])
print(features[0:20])

(2222, 623)
['win' 'wish' 'without' 'women' 'wont' 'word' 'work' 'work families'
 'workers' 'world' 'worst' 'would' 'write' 'wrong' 'year' 'years'
 'years ago' 'yesterday' 'yet' 'youre']
['able' 'able please' 'access' 'accomplish' 'achieve' 'across'
 'across nation' 'act' 'action' 'actually' 'address' 'administration'
 'affect' 'afford' 'affordable' 'ago' 'ago today' 'ahead' 'allow' 'alone']


**Comment.** We asked for at most 2,000 features but only get **623**: the binding constraint is `min_df=0.005` (a feature must appear in at least 0.5% of the tweets, i.e. about 11 tweets), and few bigrams pass this threshold ("come together", "health care", "make sure", "soul nation"...). Note also that the stop words are removed **before** the bigrams are built, so "donald trump" or "hillary clinton" disappear because their individual words are stop words (adding the bigram itself to the list has no effect). Some uninformative words remain ("would", "get", "youre"): we could add them to the stop-word list.

#### Optional: visualize your matrix as a dataframe. 

As an additional exercise and also to better visualize the matrix, you can convert the sparse matrix to a dataframe, where the column names are the features (the words). We can also add the tweet text as a column to better visualize the matrix, and the candidates name as another column.

In [10]:
# Sparse matrix -> data frame, one column per feature (word or bigram), one row per tweet
tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=features, index=df.index)

# the column names must not collide with a feature
assert not {'tweet_text', 'tweet_candidate', 'cluster', 'cluster_emb'} & set(features)
tfidf.insert(0, 'tweet_text', df['original_text'])
tfidf.insert(1, 'tweet_candidate', df['candidate'])

tfidf.head()

,tweet_text,tweet_candidate,able,able please,access,accomplish,achieve,across,across nation,act,...,worst,would,write,wrong,year,years,years ago,yesterday,yet,youre
index,,,,,,,,,,,,,,,,,,,,,
2233,Donald Trump is the most corrupt president in ...,biden,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.267276,0.258361,0.0,0.0,0.0,0.207531,0.0,0.0,0.0,0.0
2234,The power to change this country is in your ha...,biden,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2235,Here’s the truth: Donald Trump inherited a gro...,biden,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2236,WISCONSIN 🧀 VOTING ALE: If you haven’t return...,biden,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2237,I’m speaking with members of the African Ameri...,biden,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


### Exercise 3: K-means clustering - unsupervised learning with text data

For the next step, we would like to group the tweets into different clusters based on their content. This is an unsupervised learning task, where we don't have predefined labels for the tweets, but we want to discover patterns and similarities among them. (e.g. what are the x main topics that the candidates are tweeting about? Do they differ between candidates?)


To do so, we are going to use the K-means clusterng algorithm, which is a popular method for clustering data points into K distinct groups based on their features. For an introduction of k-mean clustering: https://www.youtube.com/watch?v=4b5d3muPQmA

The parameter that we provide is the K - the number of clusters that we want, and we will see how the tweets are grouped into these clusters.

Please run the next block

In [11]:
# Define the quantity of clusters - try around with this number! 
k = 3

# apply the Kmeans algorithm
kmeans = KMeans(n_clusters=k, random_state=2025) # WHy do we have to set a seed? which step in the algorithm is random?
kmeans.fit(tfidf_matrix)

# attach the labels to the dataframe 
tfidf['cluster'] = kmeans.labels_

# Check out how many tweets in each cluster
tfidf['cluster'].value_counts()

cluster
1    1646
2     357
0     219
Name: count, dtype: int64

**Why set a seed?** K-means is a randomized algorithm: the **initial position of the k centroids** is drawn at random (with the default `init='k-means++'`, the first centroid is a random data point and the next ones are drawn at random with probability proportional to their squared distance to the closest existing centroid). The algorithm then alternates between "assign each point to the closest centroid" and "move each centroid to the mean of its points", and it converges to a **local** optimum that depends on this starting point. Different seeds can therefore give different clusters (and different cluster numbers). With `random_state` fixed, the results can be reproduced.

Now that we have just imposed the K-mean clustering to the Tf-idf vectorizer, it is still not very clear what each cluster means. To better understand the clusters, we can look at the top terms (words) that are most representative of each cluster. we are also interested in which candidate tweets belong to which cluster.

Please run the next block. what are the three vectors in the output? How can you find the corresponding words? 

In [12]:
# top terms of each cluster. What do these numbers mean? 
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
order_centroids


array([[544, 358,  92, ..., 451, 177, 311],
       [362, 111, 215, ..., 497, 332, 218],
       [328, 542, 257, ..., 293, 100, 311]], shape=(3, 623))

**What are the three vectors?** `kmeans.cluster_centers_` has shape (3, 623): one row per cluster, the coordinates of its **centroid** in the tf-idf space (the average tf-idf vector of the tweets in the cluster). `argsort()[:, ::-1]` sorts each row in **decreasing** order and returns the **indices** of the features, not the values. So the first number of each row is the index of the feature with the highest weight in that cluster's centroid, the second one is the next highest, and so on. To get the words, we look these indices up in `tfidf_vectorizer.get_feature_names_out()` (e.g. `terms[544]`), as in the next cell.

In [13]:
# Each row of order_centroids: the indices of the features, sorted by decreasing weight in the centroid.
# The index -> word mapping is given by get_feature_names_out()
terms = tfidf_vectorizer.get_feature_names_out()
n_top = 15
for i in range(k):
    top_terms = [terms[ind] for ind in order_centroids[i, :n_top]]
    print(f"Cluster {i} ({(kmeans.labels_ == i).sum()} tweets): {', '.join(top_terms)}")

# Which candidate tweets belong to which cluster (only Biden here, since we cut the data)
pd.crosstab(tfidf['cluster'], tfidf['tweet_candidate'])

Cluster 0 (219 tweets): together, nation, come, come together, soul, battle, stronger, soul nation, get, defeat, unite, stand, make, battle soul, emerge
Cluster 1 (1646 tweets): need, covid, get, take, care, let, every, americans, work, crisis, health, job, right, live, say
Cluster 2 (357 tweets): make, today, help, early, get, plan, folks, better, register, head, back, chip, ballot, need, make sure


tweet_candidate,biden
cluster,
0,219
1,1646
2,357


The three vectors in the output represent the centroids of the three clusters formed by the K-means algorithm. Each centroid is a vector in the same feature space as the original data points (tweets), and it represents the "center" of each cluster. The values in each centroid vector correspond to the importance of each feature (word) in that cluster.

Do you think that the clusters make sense? Can you interpret what each cluster is about based on the top terms?

In [14]:
# Look at a few tweets in each cluster to interpret them
pd.set_option('display.max_colwidth', 150)
for i in range(k):
    print(f"--- Cluster {i}")
    for text in tfidf.loc[tfidf['cluster'] == i, 'tweet_text'].sample(5, random_state=1):
        print("  *", text)

--- Cluster 0
  * We will get through this, and we will come out stronger. I promise. 
  * If we do our part, if we stand together, if we keep faith with the past and with each other — then the divisions of our time can give way to the dreams of a brighter, better future. That is our work. That is our pledge. That is our mission.
  * We’ve got to come together to defeat Donald Trump –– Democrats, Independents, Republicans, and yes, even Demo-cats. 
  * I know the American people will beat this. We’re brave and big-hearted. We’ve emerged stronger from every crisis that has tested us as a nation — and we will again.
  * You can't defeat bigotry; it only hides. And when leaders give it oxygen — as Donald Trump has done — it comes roaring back.We all have a moral obligation to stand up, speak out, and hold people accountable. 
--- Cluster 1
  *  The future of health care, our climate, our education system and so much more is at stake in this election. That's why i…
  *  We want a president

**Do the clusters make sense?** Partly:

- **Cluster 0 (219 tweets): "unity / soul of the nation"**: *together, nation, come together, soul, battle, stronger, unite, defeat, emerge*. A coherent, recognizable theme: Biden's "battle for the soul of the nation" message and his calls for unity.
- **Cluster 2 (357 tweets): "get out the vote / campaign logistics"**: *today, early, plan, register, head, chip (in), ballot, make sure*. Mobilization: register, make a voting plan, vote early, donate.
- **Cluster 1 (1,646 tweets): everything else**: *need, covid, take, care, work, crisis, health, job, right*. Policy content (COVID, health care, jobs) mixed with all the tweets that do not fit the other two clusters.

This pattern is typical of k-means on tf-idf with short texts. Tweets are very sparse (a few non-zero features out of 623), so most of them are close to each other near the origin and end up in one large catch-all cluster. Only groups with a very specific shared vocabulary ("come together", "vote early") form clean clusters. Trying other values of k (5, 8, 10...) splits the big cluster into more interpretable topics. Here all tweets are Biden's, so we cannot compare candidates; to do so we would need to fit on the full data and cross-tabulate clusters with candidates.

### Exercise 4: Getting to Know Embeddings

TfIDf is a simple and effective way to convert text data into a numerical format that can be used for machine learning tasks. However, it has some limitations, such as not capturing the semantic meaning of words and phrases; "happy" and "joyful" are treated as completely different words, even though they have similar meanings. We might want to associate words that are similar in meaning and make the model "understand" the text better.

Entering: embeddings. Embeddings are dense vector representations of words or phrases that capture their semantic meaning and relationships. Popular embedding techniques include Word2Vec, GloVe, and FastText. These embeddings are typically pre-trained on large corpora of text data and can be fine-tuned for specific tasks. We are going to download a pre-trained word2vec model and use it to convert the tweets into embeddings.
Please run the next block.

In [ ]:
# import a tweet embedding model
import gensim.downloader
glove_vectors = gensim.downloader.load('glove-twitter-25')

The model that we download here, is called "KeyedVectors". What could that possibly mean? 

Please run the next block. Here, we are asking the model to provide the vector representation for a specific word (in this case, "happy"). 

**"KeyedVectors"**: the model is basically a lookup table (a dictionary). Each **key** is a word (a token of the Twitter vocabulary the model was trained on, about 1.2 million words) and it is mapped to a **vector** (here 25 numbers). It contains only the trained word vectors, without the training algorithm, so it cannot be trained further: we can only look up words and compute distances between their vectors. A word that is not a key (out-of-vocabulary) has no vector.

In [ ]:
glove_vectors['election']

The model represents the word in an abstract 25-dimensional space. Each dimension captures some latent semantic feature of the word, and the values in the vector represent the strength or presence of those features. 

For example, imagine that one of the dimensions captures the concept of "positivity" or "happiness". A word like "happy" might have a high value in this dimension, indicating that it is strongly associated with positive emotions. On the other hand, a word like "sad" might have a low value in this dimension, indicating that it is associated with negative emotions. 

The pre-trained embeddings are learned from a large corpus, and we can exploit some cool functions that the model has. For example, please try the subsequent functions: 

- use the most_similar function to find the words that are most similar to "election". 
- use the similarity function to test the similarity between the following group of words: 'happy' and 'joy', 'happy' and 'sad', 'happy' and 'bus'; then, try with some other topics (e.g. immigration vs politics)

What can you say about the similarity features? Do they make sense? In what case do they not perform well? 

In [ ]:
# Words most similar to "election" (cosine similarity in the 25-dimensional space)
print(glove_vectors.most_similar('election', topn=10))

pairs = [('happy', 'joy'), ('happy', 'sad'), ('happy', 'bus'),
         ('immigration', 'politics'), ('immigration', 'border'), ('immigration', 'wall'),
         ('vote', 'ballot'), ('democrat', 'republican'), ('trump', 'biden'), ('love', 'hate')]
for w1, w2 in pairs:
    print(f"{w1:12} vs {w2:12}: {glove_vectors.similarity(w1, w2):.3f}")

**What can we say about the similarities?** `most_similar` and `similarity` use the **cosine similarity** between word vectors (1 = same direction, 0 = unrelated). GloVe learns these vectors from **co-occurrence**: two words are close if they appear in similar contexts.

- It works well for related words: the neighbours of "election" are other election words (other forms of the word, voting, campaign, political terms), and pairs like *vote/ballot* or *immigration/border* come out with a high similarity. *happy/joy* is high and *happy/bus* is much lower, as expected.
- Its main limit: **similar context ≠ similar meaning**. *happy/sad*, *love/hate*, *democrat/republican* or *trump/biden* often come out **very similar**, sometimes as similar as synonyms, because antonyms and opposite sides appear in the same contexts ("I am so ___ today", "the ___ candidate"). The embedding knows that words belong to the same topic, not whether they have the same meaning or polarity. So it is a poor tool for sentiment or for telling political sides apart.
- Other limits: one vector per word whatever its meaning ("trump" the person vs. the verb, "fair" just vs. fair the event); words absent from the vocabulary get no vector; the vectors reflect the period and the population of the Twitter users the model was trained on (including their biases); and 25 dimensions is very small, so the similarities are noisy (larger models such as `glove-twitter-200` are more precise).

### Exercise 5: Word embeddings for tweets

Now, we would like to use our pre-made embedding model to convert the tweets into embeddings. Here, since we have multiple words in a tweet, we will do something simple: take the average of the embeddings of all the words in the tweet. (We can do more complicated things: check out sentence embeddings such as BERT!

Please run the next block. Explain the size of the outcome object "embeddings". What does each vector in this represent? 

In [ ]:

# Get the embeddings for each tweet by averaging the word vectors
def get_tweet_embedding(tweet):
    words = word_tokenize(tweet)
    word_vectors = [glove_vectors[word] for word in words if word in glove_vectors]
    if word_vectors:
        return np.mean(word_vectors, axis=0)
    else:
        return np.zeros(glove_vectors.vector_size)
embeddings = np.array([get_tweet_embedding(tweet) for tweet in df['processed_text']])
embeddings.shape

**Size of `embeddings`**: **(2222, 25)**, one row per Biden tweet (2,222 tweets) and one column per dimension of the GloVe space (25). Each row is the **average of the GloVe vectors of the words of the tweet** that are in the model's vocabulary, so it locates the tweet in the same semantic space as the words (a tweet that is all zeros had no known word). Unlike the tf-idf matrix (2222 × 623, sparse, one column per word), this matrix is small and dense, and its columns have no direct interpretation.

Next, we will apply K-mean clustering to the embeddings and see how the tweets are grouped into different clusters. Please run the next block.Please adapt the code above the runs K means clustering for the new matrix object "embeddings". 

Comparing the two approaches, both TFIDF and the embeddings are ways to convert text data into numerical format for machine learning tasks. What are their main difference? Which one do you like and why? 

In [ ]:
# Same K-means, but on the (2222 x 25) embedding matrix
kmeans_emb = KMeans(n_clusters=k, random_state=2025)
kmeans_emb.fit(embeddings)

tfidf['cluster_emb'] = kmeans_emb.labels_
print(tfidf['cluster_emb'].value_counts())

# How do the two clusterings compare?
pd.crosstab(tfidf['cluster'], tfidf['cluster_emb'], rownames=['tf-idf cluster'], colnames=['embedding cluster'])

**TF-IDF vs. embeddings: main differences**

| | TF-IDF | Averaged word embeddings |
|---|---|---|
| Dimension | one column per word/n-gram of *our* corpus (623 here), sparse | fixed and small (25), dense |
| What is learned | nothing outside our corpus: weights come from frequencies in our tweets | pre-trained on a huge external corpus (billions of tweets) |
| Synonyms | "happy" and "joyful" are unrelated columns | close vectors, so the model "knows" they are related |
| Interpretability | each column is a word, so it is easy to explain | dimensions are abstract and cannot be read |
| Word order / specificity | bag of words (bigrams help a little); specific rare words get high weight | averaging blurs everything; frequent generic words dominate the mean |
| Out of vocabulary | every word of the corpus is a feature | words unknown to the model are ignored |

Which one do I prefer? For this task (finding *topics* in short political tweets and explaining them), I prefer **tf-idf**: the clusters can be read directly from their top words, the method is transparent, and specific campaign terms ("ballot", "covid", "soul") carry most of the information. Embeddings help when the vocabulary is varied and we care about meaning across synonyms (or as input to a supervised model). Averaging the word vectors of a whole tweet loses a lot, though, and contextual sentence embeddings (BERT-type) would do much better.

We would like to know whether the clusters formed by the word2vec embeddings are more meaningful and coherent than those formed by the tf-idf vectorizer. To do so, we want take a look at the words with highest tf-idf scores in cluster 1. ("for this clusters, here is the most frequent words adjusted with the corpus speciality...) 

- please write the code that exercise this function : that prints out the top 20 words by tfidf score in each cluster. 
- what do you think each cluster is about?
- Do you think embeddings performs necesssariily better than the tf-idf vectorizer? Why or why not?

In [ ]:
def top_tfidf_words(tfidf_df, cluster_column, feature_columns, n=20):
    "Top n features by mean tf-idf score among the tweets of each cluster"
    mean_scores = tfidf_df.groupby(cluster_column)[feature_columns].mean()
    return {cluster: list(row.nlargest(n).index) for cluster, row in mean_scores.iterrows()}

feature_columns = list(tfidf_vectorizer.get_feature_names_out())

print("=== Clusters from the embeddings")
for cluster, words in top_tfidf_words(tfidf, 'cluster_emb', feature_columns).items():
    print(f"Cluster {cluster} ({(tfidf['cluster_emb'] == cluster).sum()} tweets): {', '.join(words)}")

print("\n=== Clusters from the tf-idf matrix (for comparison)")
for cluster, words in top_tfidf_words(tfidf, 'cluster', feature_columns).items():
    print(f"Cluster {cluster} ({(tfidf['cluster'] == cluster).sum()} tweets): {', '.join(words)}")

In [ ]:
# A few tweets of each embedding cluster
for i in range(k):
    print(f"--- Embedding cluster {i}")
    for text in tfidf.loc[tfidf['cluster_emb'] == i, 'tweet_text'].sample(5, random_state=1):
        print("  *", text)

**Interpretation** (run the two cells above: the exact clusters depend on the downloaded model and the seed).

- The top-20 tf-idf words of the **embedding clusters** are usually **much less distinctive** than those of the tf-idf clusters: the same generic words (*get, need, make, every, today, help...*) show up in several clusters, and it is hard to give each cluster a clear topic. Embedding clusters tend to group tweets by **style or register** (short calls to action and campaign announcements vs. long policy/narrative tweets; tweets with very few known words, which end up near the zero vector, often form their own cluster) rather than by **subject**.
- The crosstab with the tf-idf clusters shows that the two partitions overlap only partially: the two methods do not find the same structure.

**Do embeddings necessarily perform better?** **No.** They capture semantic similarity between words, but:
1. averaging the vectors of all words in a tweet blurs the specific content: a tweet becomes a mix of generic words, and the few discriminating words are diluted;
2. the GloVe model was trained on general Twitter language, not on political language. It does not know that "chip in" means donating to a campaign, and opposite words have similar vectors;
3. 25 dimensions is a very compressed representation;
4. the resulting clusters cannot be interpreted directly, while tf-idf clusters can be described by their top words.

The better representation depends on the task: for interpretable topic discovery on short texts tf-idf is a strong baseline, and for semantic tasks better embeddings (larger GloVe, sentence/contextual embeddings such as BERT) or supervised learning on labeled data work better. In any case, unsupervised clusters must be checked by reading the tweets, since "the algorithm found 3 clusters" does not mean that there are 3 meaningful topics.

In these two classes, we learned how to convert text data into numerical format using different techniques, and how to apply unsupervised learning algorithms to discover patterns and relationships in the data. 

That being said, as we can see, the unsupervised learning is not a great techniques with such a "loose" data like tweets. The texts are short, noisy, and often lack context, making it difficult to extract meaningful information. In the next lesson, we will learn more about machine learning, particularly supervised learning, where we have predefined labels for the data and can train models to predict these labels based on the features extracted from the text. 